# 00 — Setup & Foundations

This notebook gets your environment ready and teaches the **core building blocks** that every RAG pattern in this series relies on. If you understand this notebook thoroughly, every subsequent one will make immediate sense.

## What you will learn
1. What an **embedding** is — from scratch with NumPy
2. What **cosine similarity** does and why it is used for retrieval
3. How **text chunking** works and why chunk size matters
4. What a **vector database** does under the hood
5. What a **prompt template** is and why RAG needs one
6. How to toggle between the **Claude backend** and the **local Ollama backend**
7. A tour of the **Helios Robotics** dataset that threads through all 7 notebooks

## Step 1 — Verify the environment

Run this cell first. It checks that your hardware (MPS), Claude API, and Ollama are all reachable.

In [1]:
import sys, os
print(f"Python: {sys.version}")

# ── Hardware ──────────────────────────────────────────────────────────────────
import torch
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch device: {device}")

# ── Claude API ────────────────────────────────────────────────────────────────
import anthropic
try:
    client = anthropic.Anthropic()
    msg = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=20,
        messages=[{"role": "user", "content": "Say OK"}]
    )
    print(f"Claude API: ✅  {msg.content[0].text.strip()!r}")
except Exception as e:
    print(f"Claude API: ❌  {e}")

# ── Ollama ────────────────────────────────────────────────────────────────────
import ollama
try:
    models = ollama.list()
    names = [m.model for m in models.models]
    print(f"Ollama: ✅  models available: {names}")
    if 'llama3.1:8b' not in names:
        print("  ⚠  llama3.1:8b not found. Run: ollama pull llama3.1:8b")
    if not any('vision' in n or 'llava' in n for n in names):
        print("  ℹ  No vision model found. For notebook 03 run: ollama pull llama3.2-vision")
except Exception as e:
    print(f"Ollama: ❌  {e}  (Is `ollama serve` running? Try: brew services start ollama)")

# ── ragkit ────────────────────────────────────────────────────────────────────
import sys; sys.path.insert(0, '..')
from ragkit.config import BACKEND, DEVICE
print(f"\nragkit BACKEND={BACKEND!r}, DEVICE={DEVICE!r}")

Python: 3.13.7 (main, Aug 14 2025, 11:12:11) [Clang 17.0.0 (clang-1700.0.13.3)]
PyTorch device: mps
Claude API: ❌  "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"
Ollama: ✅  models available: ['llama3.1:8b']
  ℹ  No vision model found. For notebook 03 run: ollama pull llama3.2-vision

ragkit BACKEND='claude', DEVICE='mps'


In [2]:
! ollama pull llama3.1:8b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling 667b0c1932bc:   0% ▕                  ▏ 3.3 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   0% ▕                  ▏ 5.5 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   0% ▕                  ▏  14 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   0% ▕                  ▏  19 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   0% ▕                  ▏  23 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   1% ▕                  ▏  26 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   1% ▕                  ▏  39 MB/4.9 GB                  pulling manifest 
pulling 66

## Step 2 — The BACKEND toggle

Every notebook in this series has this toggle cell near the top. Change it to switch between Claude and local Ollama for generation.

In [2]:
import sys; sys.path.insert(0, '..')
import ragkit.config as cfg
import os

# ╔══════════════════════════════════════════════════════╗
# ║  TOGGLE: change to "local" to run fully offline      ║
# ╚══════════════════════════════════════════════════════╝
cfg.BACKEND = "local"   # "claude" | "local"
os.environ["ANTHROPIC_API_KEY"] = "REDACTED_ANTHROPIC_KEY"
os.environ["HF_TOKEN"] = "REDACTED_HF_TOKEN"
print(f"Using backend: {cfg.BACKEND!r}")
print("  claude → generation via Anthropic API (needs ANTHROPIC_API_KEY)")
print("  local  → generation via Ollama (fully offline, Metal-accelerated)")

Using backend: 'local'
  claude → generation via Anthropic API (needs ANTHROPIC_API_KEY)
  local  → generation via Ollama (fully offline, Metal-accelerated)


## Step 3 — What is an embedding?

An **embedding** converts a piece of text into a list of numbers (a vector) that captures its *meaning*. Similar meanings → similar vectors.

Think of it like GPS coordinates for meaning: Paris and Lyon are geographically close, just as "robot arm" and "robotic manipulator" are semantically close.

Let's build a tiny demo **without any libraries** first, just to build intuition.

In [3]:
import numpy as np

# Imagine each dimension represents a concept:
# [robotics, software, battery, temperature]
#
# We manually assign values (in reality, a neural network learns these)

toy_vectors = {
    "HeliosArm robot arm": np.array([0.9, 0.2, 0.1, 0.3]),
    "battery replacement": np.array([0.1, 0.1, 0.9, 0.2]),
    "firmware update": np.array([0.2, 0.9, 0.1, 0.1]),
    "joint overheating": np.array([0.7, 0.3, 0.1, 0.9]),
    "manipulator payload": np.array([0.8, 0.1, 0.0, 0.2]),
}

# Cosine similarity: how much do two vectors point in the same direction?
# Ranges from -1 (opposite) to +1 (identical direction)
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

query = "robot arm overheating"
query_vec = np.array([0.8, 0.2, 0.0, 0.8])  # high robotics + high temperature

print(f"Query: '{query}'")
print("-" * 50)
results = []
for text, vec in toy_vectors.items():
    sim = cosine_sim(query_vec, vec)
    results.append((sim, text))

for sim, text in sorted(results, reverse=True):
    bar = "█" * int(sim * 20)
    print(f"  {sim:.3f} {bar:<20} {text}")

print("\n→ 'joint overheating' scores highest because it shares high robotics AND high temperature.")
print("→ 'battery replacement' scores low despite being about hardware.")

Query: 'robot arm overheating'
--------------------------------------------------
  0.986 ███████████████████  joint overheating
  0.893 █████████████████    HeliosArm robot arm
  0.859 █████████████████    manipulator payload
  0.392 ███████              firmware update
  0.243 ████                 battery replacement

→ 'joint overheating' scores highest because it shares high robotics AND high temperature.
→ 'battery replacement' scores low despite being about hardware.


### Exercise — Test your embedding intuition

Work through these **before** moving to Step 4. Reuse `cosine_sim` and `toy_vectors` from the cell above (re-run it if needed).

1. **Design a vector**: The phrase `"cooling fan replacement"` is about robotics hardware and heat management, not software or batteries. Assign it a plausible 4-number vector `[robotics, software, battery, temperature]`.

2. **Predict first**: For the query `"robot needs a software upgrade"`, which **two** phrases in `toy_vectors` should rank highest? Write your prediction as a comment before running any code.

3. **Implement**: Add your new phrase to `toy_vectors`, build a query vector for the query above, and print the ranked cosine-similarity results (same loop as Step 3).

4. **Reflect**: In one sentence in a comment, explain why cosine similarity uses vector *direction* rather than raw magnitude.

In [4]:
import numpy as np

# ── Task 1: design a vector for "cooling fan replacement" ─────────────────────
cooling_fan_vec = np.array([
    # robotics, software, battery, temperature
    0.7, 0.1, 0.5, 0.7,   # ← replace with your values
])

# ── Task 2: predict top-2 matches (write before running) ──────────────────────
# Prediction for query "robot needs a software upgrade":
#   1. HeliosArm robot arm
#   2. firmware update

# ── Task 3: add phrase, embed query, rank results ─────────────────────────────
toy_vectors["cooling fan replacement"] = cooling_fan_vec

query = "robot needs a software upgrade"
query_vec = np.array([
    # robotics, software, battery, temperature
    0.9, 0.9, 0.05, 0.05,   # ← replace with your values
])

print(f"Query: '{query}'")
print("-" * 50)
results = []
for text, vec in toy_vectors.items():
    sim = cosine_sim(query_vec, vec)
    results.append((sim, text))

for sim, text in sorted(results, reverse=True):
    bar = "█" * int(sim * 20)
    print(f"  {sim:.3f} {bar:<20} {text}")

# ── Task 4: one-sentence reflection (comment) ─────────────────────────────────
# Your answer:

# ── Self-check (uncomment after completing tasks 1–3) ─────────────────────────
assert cooling_fan_vec[3] > cooling_fan_vec[1], "temperature should beat software"
assert cosine_sim(query_vec, toy_vectors["firmware update"]) > cosine_sim(query_vec, toy_vectors["battery replacement"])
top_two = [t for _, t in sorted(results, reverse=True)[:2]]
assert "firmware update" in top_two, f"expected firmware update in top 2, got {top_two}"
print("\n✅ Exercise checks passed!")

Query: 'robot needs a software upgrade'
--------------------------------------------------
  0.841 ████████████████     firmware update
  0.813 ████████████████     HeliosArm robot arm
  0.774 ███████████████      manipulator payload
  0.630 ████████████         joint overheating
  0.549 ██████████           cooling fan replacement
  0.198 ███                  battery replacement

✅ Exercise checks passed!


## Step 4 — Real embeddings with sentence-transformers

In reality, embeddings are 384–1536 dimensional and learned by a neural network on billions of text examples. `sentence-transformers` gives us access to these pre-trained models.

In [5]:
from ragkit.embeddings import embed

texts = [
    "HeliosArm V2 robotic arm joint 4 overheating issue",
    "battery replacement procedure for HeliosBase M1",
    "firmware update download and installation",
    "robot arm payload capacity specifications",
    "weather forecast for Stuttgart this weekend",  # completely unrelated
]

vecs = embed(texts)
print(f"Embedding shape: {vecs.shape}  → {len(texts)} texts, each a {vecs.shape[1]}-dim vector")
print(f"All vectors are L2-normalised: norms ≈ {np.linalg.norm(vecs, axis=1).round(3)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (5, 384)  → 5 texts, each a 384-dim vector
All vectors are L2-normalised: norms ≈ [1. 1. 1. 1. 1.]


In [8]:
from ragkit.embeddings import cosine_similarity


query = "What is wrong with Joint 4 when it is hot?"
q_vec = embed([query])[0]

sims = cosine_similarity(q_vec, vecs)

print(f"Query: '{query}'")
print("-" * 60)
for score, text in sorted(zip(sims, texts), reverse=True):
    bar = "█" * int(score * 30)
    print(f"  {score:.3f} {bar:<30} {text[:55]}")

Query: 'What is wrong with Joint 4 when it is hot?'
------------------------------------------------------------
  0.599 █████████████████              HeliosArm V2 robotic arm joint 4 overheating issue
  0.094 ██                             robot arm payload capacity specifications
  0.072 ██                             weather forecast for Stuttgart this weekend
  0.048 █                              firmware update download and installation
  0.041 █                              battery replacement procedure for HeliosBase M1


The "weather forecast" sentence scores near 0 — the model correctly identifies it as irrelevant. The joint overheating text scores highest.

## Step 5 — Text chunking

Documents are usually too long to embed as a single unit (embedding models have a token limit, and longer text produces coarser representations). We split them into **overlapping chunks** — the overlap prevents important context from being cut off at a boundary.

In [9]:
from ragkit.data import chunk_text

sample_doc = """
The HeliosArm V2 is a 6-degree-of-freedom industrial robotic arm designed for precision assembly, welding, 
and material handling. It has a payload capacity of 12 kg and a reach of 1,350 mm. Joint 4 uses a torque 
sensor that can exhibit erratic readings above 130 degrees per second at ambient temperatures exceeding 
38 degrees Celsius. The firmware patch FW-V2-2.3.2 addresses this by increasing the Butterworth filter 
order and adding a temperature-compensation coefficient. Interim mitigation is to limit Joint 4 speed to 
120 degrees per second in the HeliosMaster safety configuration.
""".strip()

chunks_small = chunk_text(sample_doc, chunk_size=30, overlap=8)
chunks_large = chunk_text(sample_doc, chunk_size=60, overlap=15)

print(f"Document: {len(sample_doc.split())} words")
print(f"\nSmall chunks (30 words, 8 overlap): {len(chunks_small)} chunks")
for i, c in enumerate(chunks_small):
    print(f"  [{i}] {c[:80]}...")

print(f"\nLarge chunks (60 words, 15 overlap): {len(chunks_large)} chunks")
for i, c in enumerate(chunks_large):
    print(f"  [{i}] {c[:100]}...")

print("\n⚠ Tradeoff: small chunks = precise retrieval but may miss context.")
print("            large chunks = more context but may retrieve irrelevant information.")

Document: 89 words

Small chunks (30 words, 8 overlap): 4 chunks
  [0] The HeliosArm V2 is a 6-degree-of-freedom industrial robotic arm designed for pr...
  [1] of 12 kg and a reach of 1,350 mm. Joint 4 uses a torque sensor that can exhibit ...
  [2] degrees per second at ambient temperatures exceeding 38 degrees Celsius. The fir...
  [3] and adding a temperature-compensation coefficient. Interim mitigation is to limi...

Large chunks (60 words, 15 overlap): 2 chunks
  [0] The HeliosArm V2 is a 6-degree-of-freedom industrial robotic arm designed for precision assembly, we...
  [1] per second at ambient temperatures exceeding 38 degrees Celsius. The firmware patch FW-V2-2.3.2 addr...

⚠ Tradeoff: small chunks = precise retrieval but may miss context.
            large chunks = more context but may retrieve irrelevant information.


### Exercise — Chunking & overlap tradeoffs

You are tuning chunking for three Helios support query types. Reuse `sample_doc` and `chunk_text` from the cell above (re-run it if needed).

| Query type | Example question | What a good chunk must contain |
|---|---|---|
| **Spec lookup** | "What is the payload capacity?" | `payload capacity` **and** `12 kg` in the same chunk |
| **Root-cause** | "Which firmware patch fixes Joint 4 temperature errors?" | `Joint 4` **and** `FW-V2-2.3.2` in the same chunk |
| **Operator action** | "What interim action should operators take?" | `Interim mitigation` **and** `120 degrees per second` in the same chunk |

1. **Predict first**: For each query type, will aggressive small chunks (`chunk_size=22, overlap=0`) work, or do you need larger chunks / more overlap? Write predictions as comments before running code.

2. **Audit configs**: Complete `audit_config` below and run it on the candidate settings. Which configs pass all three query types? Which pass only one?

3. **Choose per scenario**: Fill in `BEST_CONFIG` for three deployment profiles — a spec-only FAQ bot, an incident-investigation bot, and a general knowledge base. Each choice should *pass* its primary query audit and you should justify the chunk count tradeoff in a comment.

4. **Reflect**: When is increasing **overlap** the right fix vs increasing **chunk_size**? One sentence each in comments.

In [11]:
from ragkit.data import chunk_text

# Phrases that must land in the SAME chunk for retrieval to succeed
QUERY_REQUIREMENTS = {
    "spec": ["payload capacity", "12 kg"],
    "root_cause": ["Joint 4", "FW-V2-2.3.2"],
    "operator": ["Interim mitigation", "120 degrees per second"],
}

def audit_config(chunk_size: int, overlap: int) -> dict:
    """Return per-query pass/fail and total chunk count for a setting."""
    chunks = chunk_text(sample_doc, chunk_size=chunk_size, overlap=overlap)
    results = {"chunk_size": chunk_size, "overlap": overlap, "num_chunks": len(chunks)}
    for name, phrases in QUERY_REQUIREMENTS.items():
        results[name] = any(all(p in chunk for p in phrases) for chunk in chunks)
    return results

def print_audit(label: str, result: dict) -> None:
    flags = " ".join(
        f"{name}:{'✅' if result[name] else '❌'}"
        for name in QUERY_REQUIREMENTS
    )
    print(f"{label:28s}  chunks={result['num_chunks']}  {flags}")

# ── Task 1: predictions (write before running) ────────────────────────────────
# Spec lookup with (22, 0): Failure, trimmed without context. Also, too long chunk size
# Root-cause with (30, 8): Bingo! Success
# Operator action with (10, 0): Q3 succeeds, values are in the same chunk

# ── Task 2: audit candidate configs ───────────────────────────────────────────
CANDIDATES = [
    (10, 0),   # tiny chunks, no overlap — many fragments
    (22, 0),   # small chunks, no overlap
    (22, 10),  # small chunks, generous overlap
    (30, 8),   # same as Step 5 "small" demo
    (60, 15),  # same as Step 5 "large" demo
]

print("Config audits on sample_doc")
print("-" * 60)
for cs, ov in CANDIDATES:
    print_audit(f"({cs}, {ov})", audit_config(cs, ov))

# ── Task 3: pick best config per deployment profile ─────────────────────────────
BEST_CONFIG = {
    # Primary query: spec lookup — prefer precise, small chunks
    "spec_faq_bot": {"chunk_size": 10, "overlap": 2},          # ← fill in
    # Primary query: root-cause — cause + fix must survive boundaries
    "incident_bot": {"chunk_size": 60, "overlap": 15},          # ← fill in
    # Primary query: all three — balanced default for mixed traffic
    "general_kb": {"chunk_size": 60, "overlap": 15},            # ← fill in
}

# Why this chunk count tradeoff? (comment per profile)
# spec_faq_bot: 10 chunks are enough to have the data in one chunk, but with 25% overlap, we can have more context
# incident_bot: 10 chunks are enough to have the data in one chunk, but with 30% overlap, we can have more context
# general_kb: Generally fair chunk_size given the length of the document, with 25% overlap

print("\nYour deployment choices")
print("-" * 60)
for profile, cfg in BEST_CONFIG.items():
    primary = {
        "spec_faq_bot": "spec",
        "incident_bot": "root_cause",
        "general_kb": "operator",  # must also pass root_cause for general_kb
    }[profile]
    result = audit_config(**cfg)
    print_audit(profile, result)

# ── Task 4: overlap vs chunk_size (comments above) ────────────────────────────

# ── Self-check (uncomment after completing tasks 2–3) ───────────────────────────
assert not audit_config(22, 0)["spec"], "(22, 0) should split spec facts across chunks"
assert audit_config(22, 10)["spec"], "overlap should reunite spec facts for small chunks"
assert not audit_config(30, 8)["root_cause"], "(30, 8) splits Joint 4 from firmware patch"
assert audit_config(60, 15)["root_cause"], "larger chunks keep cause + fix together"
for profile, primary in [("spec_faq_bot", "spec"), ("incident_bot", "root_cause")]:
    cfg = BEST_CONFIG[profile]
    assert audit_config(**cfg)[primary], f"{profile} must pass {primary} audit"
assert audit_config(**BEST_CONFIG["general_kb"])["root_cause"], "general_kb needs root_cause too"
print("\n✅ Exercise checks passed!")

Config audits on sample_doc
------------------------------------------------------------
(10, 0)                       chunks=9  spec:✅ root_cause:❌ operator:❌
(22, 0)                       chunks=5  spec:❌ root_cause:❌ operator:✅
(22, 10)                      chunks=7  spec:✅ root_cause:❌ operator:❌
(30, 8)                       chunks=4  spec:✅ root_cause:❌ operator:✅
(60, 15)                      chunks=2  spec:✅ root_cause:✅ operator:✅

Your deployment choices
------------------------------------------------------------
spec_faq_bot                  chunks=11  spec:✅ root_cause:❌ operator:❌
incident_bot                  chunks=2  spec:✅ root_cause:✅ operator:✅
general_kb                    chunks=2  spec:✅ root_cause:✅ operator:✅

✅ Exercise checks passed!


### Exercise — Chunking across document span (RuPaul's Drag Race)

This exercise uses the same `audit_config` pattern but with a trickier document: one query pair (`network_move`) spans the **entire document** — first sentence to last. That gap cannot be fixed by increasing overlap alone.

| Query type | Example question | What a good chunk must contain |
|---|---|---|
| **debut** | "When and where did Drag Race premiere?" | `February 2, 2009` **and** `Logo TV` in the same chunk |
| **prize** | "What does the season winner receive?" | `America's Next Drag Superstar` **and** `$100,000` in the same chunk |
| **network_move** | "Which network did the show move to?" | `Logo TV` **and** `VH1` in the same chunk |

1. **Predict first**: For `(22, 0)`, `(30, 8)`, and `(60, 15)` — which query types pass and which fail? Write predictions as comments before running.

2. **Audit configs**: Run the five candidate settings. Which single config passes all three?

3. **Choose per profile**: Fill in `BEST_CONFIG` for three bots — a trivia bot (debut only), a history bot (must pass network_move), and a general knowledge base (all three). Add a one-line comment per profile justifying the chunk count tradeoff.

4. **Reflect**: `network_move` fails at `(60, 15)` even though that same size unlocked `root_cause` in the Helios exercise above. In one sentence (comment): why does the required chunk size differ between the two documents?

In [8]:
from ragkit.data import chunk_text

# ~77 words: "Logo TV" at word 10, "VH1" at word 74 — a 64-word gap.
# (60, 15) produces two chunks that each hold only one of the pair.
# (80, 20) exceeds the document length → one chunk → both phrases coexist.
rupaul_doc = """
RuPaul's Drag Race debuted on February 2, 2009 on Logo TV.
RuPaul Charles hosts the show with World of Wonder Productions.
Contestants compete in weekly Maxi and Mini Challenges for a celebrity panel.
The bottom two queens lip sync; the loser must sashay away.
The winner earns America's Next Drag Superstar title and $100,000.
Season 14's Willow Pill beat Angeria Paris VanMichaels in the finale.
Starting with Season 9, the show moved to VH1 for broader reach.
""".strip()

RUPAUL_REQUIREMENTS = {
    "debut":        ["February 2, 2009", "Logo TV"],
    "prize":        ["America's Next Drag Superstar", "$100,000"],
    "network_move": ["Logo TV", "VH1"],
}

def audit_rupaul(chunk_size: int, overlap: int) -> dict:
    chunks = chunk_text(rupaul_doc, chunk_size=chunk_size, overlap=overlap)
    results = {"chunk_size": chunk_size, "overlap": overlap, "num_chunks": len(chunks)}
    for name, phrases in RUPAUL_REQUIREMENTS.items():
        results[name] = any(all(p in chunk for p in phrases) for chunk in chunks)
    return results

def print_rupaul_audit(label: str, result: dict) -> None:
    flags = "  ".join(
        f"{name}:{'✅' if result[name] else '❌'}" for name in RUPAUL_REQUIREMENTS
    )
    print(f"{label:20s}  chunks={result['num_chunks']}  {flags}")

# ── Task 1: predictions (write before running) ────────────────────────────────
# (22, 0)  debut: ?  prize: ?  network_move: ?
# (30, 8)  debut: ?  prize: ?  network_move: ?
# (60, 15) debut: ?  prize: ?  network_move: ?

# ── Task 2: audit candidate configs ───────────────────────────────────────────
CANDIDATES = [(10, 0), (22, 0), (30, 8), (60, 15), (80, 20)]

print("Config audits on rupaul_doc")
print("-" * 65)
for cs, ov in CANDIDATES:
    print_rupaul_audit(f"({cs}, {ov})", audit_rupaul(cs, ov))

# ── Task 3: pick best config per deployment profile ───────────────────────────
RUPAUL_BEST = {
    # Primary: debut — both phrases in sentence 1
    "trivia_bot":  {"chunk_size": 22, "overlap": 0},   # ← fill in
    # Primary: network_move — Logo TV (sentence 1) and VH1 (sentence 7) must coexist
    "history_bot": {"chunk_size": 80, "overlap": 20},  # ← fill in
    # Primary: all three
    "general_kb":  {"chunk_size": 80, "overlap": 20},   # ← fill in
}

# Why this chunk count tradeoff? (one comment per profile)
# trivia_bot:
# history_bot:
# general_kb:

print("\nDeployment choices")
print("-" * 65)
for profile, cfg in RUPAUL_BEST.items():
    print_rupaul_audit(profile, audit_rupaul(**cfg))

# ── Task 4: reflection (comment) ──────────────────────────────────────────────
# network_move needs a larger chunk_size than root_cause because:

# ── Self-check ────────────────────────────────────────────────────────────────
assert audit_rupaul(22, 0)["debut"],           "(22, 0) should keep debut phrases (same sentence) together"
assert not audit_rupaul(22, 0)["network_move"], "small chunks can't bridge sentence 1 to sentence 7"
assert not audit_rupaul(60, 15)["network_move"], "(60, 15) still too small for the full document span"
assert audit_rupaul(**RUPAUL_BEST["trivia_bot"])["debut"],         "trivia_bot must pass debut"
assert audit_rupaul(**RUPAUL_BEST["history_bot"])["network_move"], "history_bot must pass network_move"
assert audit_rupaul(**RUPAUL_BEST["general_kb"])["network_move"],  "general_kb must pass network_move"
assert audit_rupaul(**RUPAUL_BEST["general_kb"])["prize"],         "general_kb must pass prize"
print("\n✅ Exercise checks passed!")

Config audits on rupaul_doc
-----------------------------------------------------------------
(10, 0)               chunks=8  debut:❌  prize:❌  network_move:❌
(22, 0)               chunks=4  debut:✅  prize:✅  network_move:❌
(30, 8)               chunks=4  debut:✅  prize:✅  network_move:❌
(60, 15)              chunks=2  debut:✅  prize:✅  network_move:❌
(80, 20)              chunks=1  debut:✅  prize:✅  network_move:✅

Deployment choices
-----------------------------------------------------------------
trivia_bot            chunks=4  debut:✅  prize:✅  network_move:❌
history_bot           chunks=1  debut:✅  prize:✅  network_move:✅
general_kb            chunks=1  debut:✅  prize:✅  network_move:✅

✅ Exercise checks passed!


## Step 6 — What is a vector database?

A vector database stores embeddings (vectors) alongside the original text and lets you quickly find the *k* most similar vectors to a query. It uses approximate nearest-neighbour (ANN) algorithms (HNSW, IVF) instead of exhaustive search — so it works even with millions of documents.

We use **ChromaDB** which runs in-process (no server needed).

In [9]:
import chromadb
from ragkit.embeddings import embed

# Build a tiny in-memory collection with 5 sentences
client_c = chromadb.EphemeralClient()
collection = client_c.create_collection("demo", metadata={"hnsw:space": "cosine"})

docs = [
    "Joint 4 torque sensor reports erratic readings above 130 degrees per second at high temperature.",
    "The HeliosBase M1 battery lasts 8 hours under typical load conditions.",
    "Firmware FW-V2-2.3.2 adds a temperature-compensation coefficient for Joint 4.",
    "The HC-400 controller runs HeliosMaster v5.x on Ubuntu 22.04 with a real-time kernel.",
    "Battery SOC drift occurs after 500 charge cycles due to capacity fade.",
]
ids = [f"doc_{i}" for i in range(len(docs))]
vecs = embed(docs).tolist()

collection.add(ids=ids, documents=docs, embeddings=vecs)
print(f"Collection has {collection.count()} documents")

# Query
query = "What is the fix for the Joint 4 temperature problem?"
q_vec = embed([query]).tolist()

results = collection.query(query_embeddings=q_vec, n_results=3,
                           include=["documents", "distances"])
print(f"\nQuery: '{query}'")
print("-" * 60)
for i, (doc, dist) in enumerate(zip(results["documents"][0], results["distances"][0])):
    print(f"  {i+1}. similarity={1-dist:.3f}  {doc[:80]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Collection has 5 documents

Query: 'What is the fix for the Joint 4 temperature problem?'
------------------------------------------------------------
  1. similarity=0.631  Firmware FW-V2-2.3.2 adds a temperature-compensation coefficient for Joint 4.
  2. similarity=0.624  Joint 4 torque sensor reports erratic readings above 130 degrees per second at h
  3. similarity=0.106  Battery SOC drift occurs after 500 charge cycles due to capacity fade.


## Step 7 — What is a prompt template?

A prompt template structures the information we send to the LLM. In RAG, the template has three parts:
1. **System instructions** — tell the model its role and constraints
2. **Context** — the retrieved chunks
3. **Question** — the user's original query

In [ ]:
RAG_SYSTEM = """You are a helpful technical assistant for Helios Robotics.
Answer questions using ONLY the provided context.
If the answer is not in the context, say "I don't have enough information."
Be concise and precise."""

RAG_TEMPLATE = """Context from the Helios knowledge base:

{context}

---
Question: {question}

Answer based only on the context above:"""

# Simulate what a RAG pipeline does:
retrieved_chunks = [
    "Joint 4 torque sensor reports erratic readings above 130 degrees per second at high temperature.",
    "Firmware FW-V2-2.3.2 adds a temperature-compensation coefficient for Joint 4.",
]
question = "What is the fix for the Joint 4 temperature problem?"

context = "\n".join(f"[{i+1}] {c}" for i, c in enumerate(retrieved_chunks))
prompt = RAG_TEMPLATE.format(context=context, question=question)

print("=== SYSTEM ===")
print(RAG_SYSTEM)
print("\n=== USER PROMPT ===")
print(prompt)

In [ ]:
# Now actually send it to the LLM
from ragkit.llm import generate

answer = generate(prompt, system=RAG_SYSTEM)
print("Answer:")
print(answer)

## Step 8 — Tour the Helios Robotics dataset

This dataset is used across all 7 notebooks. It's designed so that:
- Some questions require finding specific part numbers (good for testing exact-match / hybrid retrieval)
- Some questions require connecting multiple documents (good for graph RAG)
- Some questions have images (good for multimodal RAG)

In [ ]:
from ragkit.data import load_corpus, load_images
from collections import Counter

corpus = load_corpus()
print(f"Corpus: {len(corpus)} documents")
cats = Counter(d['category'] for d in corpus)
for cat, count in sorted(cats.items()):
    print(f"  {cat:10s}: {count} doc(s)")

print()
for doc in corpus:
    words = len(doc['text'].split())
    print(f"  {doc['source']:45s}  {words:4d} words")

In [ ]:
images = load_images()
print(f"Images: {len(images)} files")
for img in images:
    print(f"  {img['filename']:45s}  {img['caption'][:70]}...")

In [ ]:
# Display the images
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, len(images), figsize=(5 * len(images), 4))
if len(images) == 1:
    axes = [axes]
for ax, img_info in zip(axes, images):
    img = Image.open(img_info['path'])
    ax.imshow(img)
    ax.set_title(img_info['filename'].replace('_', ' ').replace('.png', ''), fontsize=7)
    ax.axis('off')
plt.suptitle("Helios Robotics Dataset — Images", fontsize=10, y=1.01)
plt.tight_layout()
plt.show()

## Step 9 — Benchmark queries used across all notebooks

These queries are designed to stress-test different RAG patterns. Keep them in mind as you go through the series — you'll see how different approaches handle the same questions.

In [ ]:
benchmark_queries = [
    # ── Factual, single-document ───────────────────────────────────────────────
    ("spec",     "What is the exact part number for the HeliosArm V2 gripper?"),
    ("spec",     "What is the payload capacity of HR-MOB-M1-AMR?"),
    # ── Exact code / part number (hard for pure vectors) ─────────────────────
    ("code",     "What does part number HR-REED-UPGRADE fix?"),
    ("code",     "Which firmware version is FW-V2-2.3.2 and what does it address?"),
    # ── Multi-hop (requires linking 2+ documents) ─────────────────────────────
    ("multihop", "Who is responsible for the firmware fix for the Joint 4 heat issue?"),
    ("multihop", "What project will permanently fix the Joint 4 problem and who leads it?"),
    # ── Procedure + specification ─────────────────────────────────────────────
    ("proc",     "How long does a HeliosBase M1 battery replacement take and what tools do I need?"),
    # ── Image-relevant ────────────────────────────────────────────────────────
    ("image",    "What are the safety zones shown in the HeliosBase M1 diagram?"),
]

print("Benchmark queries (used across all 7 notebooks):")
print()
for i, (typ, q) in enumerate(benchmark_queries, 1):
    print(f"  {i}. [{typ:8s}] {q}")

## Summary

| Concept | What it is | Implemented in |
|---|---|---|
| Embedding | Text → fixed-size vector capturing meaning | `ragkit/embeddings.py` |
| Cosine similarity | Angle between two vectors (1 = same direction) | `ragkit/embeddings.py` |
| Chunking | Split long doc into overlapping windows | `ragkit/data.py` |
| Vector DB (Chroma) | Fast ANN lookup of similar vectors | `ragkit/vectorstore.py` |
| Prompt template | System + context + question → LLM input | Inline in notebooks |
| Backend toggle | `cfg.BACKEND = "claude"` or `"local"` | `ragkit/config.py` |

**Next:** [01_naive_rag.ipynb](01_naive_rag.ipynb) — build a complete RAG pipeline end-to-end.